# AeroStruct-15MW — End-to-End Workflow

This notebook is the cleaned public entry point for the completed project. It uses the modular `src/aerostruct15mw` package. The original exploratory Colab notebook is preserved separately as `00_full_development.ipynb`.

The workflow is reduced-order and is **not** an IEC/OpenFAST certification study.

## 1. Environment and public reference data

From the repository root, install the package and fetch the public IEA reference inputs once.

In [ ]:
# Run from notebooks/
%pip install -e ..
!python ../scripts/fetch_reference_data.py

## 2. Load geometry and structural properties

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from aerostruct15mw.config import TurbineConfig, FinalDesign
from aerostruct15mw.io import load_processed_reference_data
ROOT = Path('..').resolve()
geometry, structural = load_processed_reference_data(ROOT / 'data' / 'processed')
polar_folder = ROOT / 'data' / 'reference' / 'airfoils'
geometry.head()

## 3. Reproduce the BEM baseline

The baseline solver uses Prandtl hub/tip losses, a Buhl-type high-induction correction, and station-specific AeroDyn polars.

In [ ]:
from aerostruct15mw.bem import evaluate_bem_geometry
cfg = TurbineConfig()
bem_base, summary_base = evaluate_bem_geometry(geometry, polar_folder, label='IEA baseline', wind_speed_mps=cfg.rated_wind_mps, rotor_rpm=cfg.rpm_rated, pitch_deg=0.0, config=cfg)
summary_base

## 4. Map structural properties and solve the rotating beam response

In [ ]:
from aerostruct15mw.structures import centrifugal_tension_from_rpm, solve_rotating_beam_fem
span = geometry['span_m'].to_numpy()
struct_span = structural['span_fraction'].to_numpy() * geometry['span_m'].max()
flap_EI = np.interp(span, struct_span, structural['flap_EI_Nm2'])
edge_EI = np.interp(span, struct_span, structural['edge_EI_Nm2'])
mass_density = np.interp(span, struct_span, structural['mass_density_kgpm'])
Ncf = centrifugal_tension_from_rpm(span, mass_density, cfg.rpm_rated, cfg.rotor_radius_m)
struct_base = solve_rotating_beam_fem(span, bem_base['normal_Npm'], flap_EI, Ncf)
struct_base

## 5. Reconstruct the final physics-verified design

The four-variable search selected approximately 1.499° smooth outboard twist and almost zero structural reinforcement.

In [ ]:
from aerostruct15mw.design import build_aerostruct_candidate
fd = FinalDesign()
final_candidate = build_aerostruct_candidate(geometry, span, flap_EI, mass_density, fd.peak_twist_deg, fd.aero_start_m, fd.struct_start_m, fd.ei_increase_percent)
print('Added mass [kg/blade]:', final_candidate.added_mass_kg)

## 6. Rated-point final candidate

In [ ]:
bem_final, summary_final = evaluate_bem_geometry(final_candidate.geometry, polar_folder, label='Final optimized', wind_speed_mps=cfg.rated_wind_mps, rotor_rpm=cfg.rpm_rated, pitch_deg=0.0, config=cfg)
Ncf_final = centrifugal_tension_from_rpm(span, final_candidate.mass_density, cfg.rpm_rated, cfg.rotor_radius_m)
struct_final = solve_rotating_beam_fem(span, bem_final['normal_Npm'], final_candidate.ei, Ncf_final)
pd.DataFrame({'baseline':[summary_base['Cp'],summary_base['root_moment_MNm'],struct_base['tip_deflection_m']], 'optimized':[summary_final['Cp'],summary_final['root_moment_MNm'],struct_final['tip_deflection_m']]}, index=['Cp','root_moment_MNm','rotating_tip_m'])

## 7. Recompute the controlled 4–25 m/s envelope

The repository script reproduces the final variable-speed / pitch-controlled envelope and saves figures/tables.

In [ ]:
!python ../scripts/run_final_candidate.py

## 8. Optional surrogate-assisted optimization

The full ML workflow generates a 300-point Latin Hypercube physics database, compares three tree-based surrogate families, evaluates a 100,000-point search, applies engineering constraints, and forms a core Pareto set. Use `--verify-pareto` to re-run the surrogate Pareto candidates through the physics evaluator.

In [ ]:
# This can take several minutes. Uncomment when you want to reproduce the optimization.
# !python ../scripts/run_ml_pipeline.py --verify-pareto

## Interpretation

The final candidate is a **reduced-order design hypothesis**. Its approximately 5.65% rated root-moment reduction and 7.23% rated rotating-tip reduction are conditional on the BEM/beam/controller/fatigue assumptions documented in `docs/assumptions_and_limitations.md`.